# Prompt Strategy Lab — Demo

Side-by-side comparison of 9 prompt-engineering strategies on the same user prompt, using a local TinyLLaMA via Ollama.

**Prerequisites:**
1. `ollama pull tinyllama`
2. `pip install -e .[dev]` from the repo root.
3. Make sure the Ollama server is running (`ollama serve` if not auto-started).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
from prompt_strategy_lab import compare_strategies, OllamaRunner, STRATEGIES, to_records  # noqa: E402
from prompt_strategy_lab.compare import to_records

print('Available strategies:')
for name, s in STRATEGIES.items():
    print(f'  - {name}: {s.description}')

## Step 1 — Define the user prompt and the strategies to compare

Pick any prompt and any subset of strategies. Add or remove strategies and re-run.

In [ ]:
USER_PROMPT = (
    'Propose a backend code structure for an AI-powered exam studying app '
    'for SAT, GRE, and Medical Exams.'
)

STRATEGIES_TO_RUN = [
    'few_shot',
    'chain_of_thought',
    'cot_with_reflection',
    'self_consistency',
]

## Step 2 — Inspect the augmented prompts before sending them

This is what the model will actually see. Each strategy wraps the same user prompt in a different scaffold.

In [ ]:
for name in STRATEGIES_TO_RUN:
    strategy = STRATEGIES[name]
    print('=' * 80)
    print(f'STRATEGY: {name}')
    print('=' * 80)
    print(strategy.augment(USER_PROMPT)[:800])
    print('...')
    print()

## Step 3 — Run the comparison

Each strategy is sent through the same `OllamaRunner` with identical generation parameters, so the only variable is the prompt scaffold itself.

In [ ]:
runner = OllamaRunner(model='tinyllama')

results = compare_strategies(
    user_prompt=USER_PROMPT,
    strategy_names=STRATEGIES_TO_RUN,
    runner=runner,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    num_predict=600,
    context_window=2048,
)

## Step 4 — Compare numerically (latency / tokens)

In [ ]:
df = pd.DataFrame(to_records(results))
df[['strategy', 'tokens_used', 'latency_ms']].sort_values('latency_ms')

## Step 5 — Compare qualitatively (the actual responses)

In [ ]:
for r in results:
    print('=' * 80)
    print(f'{r.strategy}  |  {r.latency_ms:.0f}ms  |  {r.tokens_used} tokens')
    print('=' * 80)
    print(r.response)
    print()

## Step 6 — Save results for the README

If you want to commit a results CSV alongside the repo, use the snippet below.

In [ ]:
from pathlib import Path
import datetime as dt

results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)
stamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = results_dir / f'comparison_{stamp}.csv'
df.to_csv(out_path, index=False)
print(f'Saved {out_path}')